### Cart Pole Balancing

<img src="cartPole.png" width="600px">

In [ ]:
!pip uninstall -y gym
!pip install -U "stable-baselines3[extra]" gymnasium pygame moviepy imageio imageio-ffmpeg

In [ ]:
import gymnasium as gym
import numpy as np
import torch
import imageio
from pathlib import Path
from IPython.display import Video, display
from stable_baselines3 import PPO
from stable_baselines3.common.evaluation import evaluate_policy

In [ ]:
class CartPoleAgent():
     def __init__(self,
                  policy:str,
                  learningRate,
                  gamma,
                  gaeLambda,
                  clipRange,
                  verbose,
                  seed):
          self.environment = gym.make("CartPole-v1")
          self.policy = policy
          self.learningRate = learningRate
          self.gamma = gamma
          self.gaeLambda = gaeLambda
          self.clipRange = clipRange
          self.verbose = verbose
          self.seed = seed

          self.model = PPO(
               policy = self.policy,      
               env = self.environment,
               learning_rate = self.learningRate,   
               gamma = self.gamma,
               gae_lambda = self.gaeLambda,
               clip_range = self.clipRange,
               verbose = self.verbose,
               seed = self.seed
          )

     def envDetails(self):
          env = self.environment
          print("Observation space = ", env.observation_space)
          print("Action space = ", env.action_space)
          print("Max steps in each episode = ", env.spec.max_episode_steps)
     
     def trainPolicy(self, timesteps=50000):
          return self.model.learn(timesteps)
     
     def evaluatePolicy(self, episodes=10, deterministic=True):
          meanReward, stdReward = evaluate_policy(
               self.model,
               self.environment,
               episodes,
               deterministic
          )
          return meanReward, stdReward
     
     def recordVideo(self, filename="trained_cartpole.mp4", deterministic=True):
        video_env = gym.make("CartPole-v1", render_mode="rgb_array")
        obs, info = video_env.reset(seed=self.seed)
        frames = []
        totalReward = 0
        for step in range(500):
            frame = video_env.render()
            frames.append(frame)
            action, _ = self.model.predict(obs, deterministic=deterministic)
            obs, reward, terminated, truncated, info = video_env.step(int(action))
            totalReward += reward
            if terminated or truncated:
                frames.append(video_env.render())
                break
        video_env.close()
        imageio.mimsave(filename, frames, fps=30)
        print("Video saved as:", filename)
        print("Video episode reward:", totalReward)
        return filename
     
     def run(self, videoName="final_cartpole_agent.mp4"):
        videoPath = self.recordVideo(
            filename=videoName,
            deterministic=True
        )
        return Video(videoPath, embed=True, width=600)

In [ ]:
agent = CartPoleAgent(
     policy="MlpPolicy",
     learningRate=3e-4,
     gamma=0.99,
     gaeLambda=0.95,
     clipRange=0.2,
     verbose=1,
     seed=20
)

In [ ]:
agent.envDetails()

In [ ]:
agent.trainPolicy()

In [ ]:
agent.evaluatePolicy()

In [ ]:
agent.run()